In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 19.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.4/810.4 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 35.9 MB/s eta 0:00:00:00:0100:01


In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

# Load YOLOv11 detection model (equivalent to yolov8x.pt)
det_model = YOLO('yolo11x.pt')

# Load YOLOv11 pose estimation model (equivalent to yolov8x-pose.pt)
pose_model = YOLO('yolo11x-pose.pt')

# Input and output image paths
image_path = 'person.jpg'
save_path = 'person-pose_with_skeleton.jpg'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Function to draw pose keypoints and skeleton (COCO 17-keypoint format)
def draw_pose(image, keypoints_xy, keypoints_conf, thickness=2):
    if keypoints_xy is None or len(keypoints_xy) == 0 or keypoints_conf is None:
        return image

    # COCO 17-keypoint skeleton (edges between keypoints)
    skeleton = [
        (0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
        (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)
    ]

    for person_idx, (kpts, confs) in enumerate(zip(keypoints_xy, keypoints_conf)):
        kpts = kpts.cpu().numpy()  # Shape: (17, 2) [x, y]
        confs = confs.cpu().numpy()  # Shape: (17,) [confidence]

        # Draw keypoints
        for i, (x, y) in enumerate(kpts):
            if confs[i] > 0.5:  # Draw keypoints with sufficient confidence
                cv2.circle(image, (int(x), int(y)), 5, (0, 0, 255), -1)

        # Draw skeleton lines
        for (start, end) in skeleton:
            if confs[start] > 0.5 and confs[end] > 0.5:
                start_pt = (int(kpts[start][0]), int(kpts[start][1]))
                end_pt = (int(kpts[end][0]), int(kpts[end][1]))
                cv2.line(image, start_pt, end_pt, (255, 0, 0), thickness)

    return image

In [4]:
import matplotlib.pyplot as plt
from PIL import Image

# Load input image
image = cv2.imread(image_path)
if image is None:
    print(f"Error: Could not load image ({image_path}).")
    exit()

# Perform YOLOv8 person detection
det_results = det_model(
    image,
    conf=0.25,
    iou=0.45,
    classes=[0],  # Person class only
    device='cuda:0',
    half=True,
    verbose=False
)

# Prepare for visualization
vis_image = image.copy()
person_bboxes = []

# Extract bounding boxes from detection results
for result in det_results:
    if result.boxes is not None and result.boxes.xyxy is not None:
        boxes = result.boxes.xyxy.cpu().numpy()  # Bounding boxes (xyxy)
        person_bboxes = boxes

        # Draw bounding boxes
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            cv2.rectangle(vis_image, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(vis_image, "Person", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 3)

# Perform pose estimation with YOLOv8-pose
pose_results = pose_model(
    image,
    conf=0.25,
    iou=0.45,
    classes=[0],  # Person class only
    device='cuda:0',
    half=True,
    verbose=False
)

# Extract keypoints and confidence scores
keypoints_xy = []
keypoints_conf = []
for result in pose_results:
    if result.keypoints is not None:
        keypoints_xy = result.keypoints.xy  # Shape: (num_persons, 17, 2) [x, y]
        print(keypoints_xy)
        keypoints_conf = result.keypoints.conf  # Shape: (num_persons, 17) [conf]

# Draw pose keypoints and skeleton
vis_image = draw_pose(vis_image, keypoints_xy, keypoints_conf)

# Save output image
cv2.imwrite(save_path, vis_image)
print(f"Output image saved to {save_path}")

# Image(filename=save_path, width=600)
img = Image.open(save_path)
plt.imshow(img)
plt.axis("off")

Error: Could not load image (person.jpg).
WARNING ⚠️ 'source' is missing. Using 'source=/usr/local/lib/python3.12/dist-packages/ultralytics/assets'.
Ultralytics 8.4.16 🚀 Python-3.12.12 torch-2.9.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [ ]:
from ultralytics import YOLO

# Run inference on an image
results = pose_model(image_path)

# Process and visualize the results
for result in results:
    # Keypoints in normalized format can be accessed
    keypoints = result.keypoints.xy # XY coordinates
    conf = result.keypoints.conf    # Confidence score
    data = result.keypoints.data    # XY and confidence score together
    print(keypoints, conf, data)